# Human Coronavirus


## STEPS

The initial idea is the evaluate and curate a standard reference set for the following Human Coronavirus analsysis. 

I'll performed these steps: 

1. Download the dataset
    - The sequences were downloaded from the NCBI viruses on 09.Sep.2026.
    - No initial filters were applyed.

2. Filter genomes by length.

3. Incorporate the genotype into the metadata.

4. Create a header for the sequences.

5. Rename and filter the fasta file.

6. Generate the trees.
    - I'll run the Augur framework to generate the trees and to classify sequences without genotypes.


## 2. Filter genomes by length.

Here I'll create a function to filter and prepare the background metadata for the downstream analysis.
This function will: 

- Perform a standardisation of the dates on the dataframe to fit into the Augur framework.
- Create a new header for the sequences
    - This reader will be free of spaces
    - And also will contain this structure Accession|Country|Collection_Date
- Keep only genomes with Country and Collection Date (Year at least)
- Make the date standard for Augur (Y-MM-DD or Y-MM-XX or Y-XX-XX)
- Filter the genomes by lenth (>=80%)
- Filter out other hosts than "Homo sapiens"

In [2]:
import pandas as pd

### Setting the functions:
1. Function to standardise the format in Date column
- since the Augur pipeleni deals with uncertanty with -XX or -XX-XX, I'll make my dates to fit in this format.
 

In [3]:
from datetime import datetime

#This function test if the date is in the format
def is_date_format(text, date_format):
    try:
        datetime.strptime(text, date_format)
        return True
    except ValueError:
        return False


#this function will test each different date format until find the right one
def classify_date(test_str):

    if pd.isna(test_str):
        return None

    if is_date_format(test_str, "%Y-%m-%d"):
        return test_str

    elif is_date_format(test_str, "%Y-%m"):
        return test_str+"-XX"

    elif is_date_format(test_str, "%Y"):
        return test_str+"-XX-XX"

    else:
        return "error"
    

2. Function to run the date format and to filter the data.
- I'll put everything into a single function to decrease the amount of lines in the code.

In [4]:
def filter_data(df, ref_length, min_coverage=0.8):

    # Running the date format function
    df["Date"] = df["Collection_Date"].apply(classify_date)
    
    # creating a new column for the head
    # it will be the paste values from Accession | Country | Collection_date
    df["name"] = (
        df["Accession"].astype(str) + "|" +
        df["Country"].astype(str) + "|" +
        df["Date"].astype(str)
    )    

    # Replacing space for _
    df["name"] = df["name"].str.replace(" ", "_")
    
    # Calculating coverage
    df["coverage"] = df["Length"]/ref_length    

    # keeping only lines with country or date 
    # and coverage >= 80%
    df_filt = (
        df[df["coverage"] >= min_coverage]
        .dropna(subset=["Country","Collection_Date"])
        .query("Host == 'Homo sapiens'")
    )

    
    return df_filt


In [5]:
# how to run the funtion line by line.
# its now incorporated into the function

#metadata_229e_filt["Date"] = metadata_229e_filt["Collection_Date"].apply(classify_date)


- HCoV 229e

In [6]:
# Importing the metadata df for 229E
metadata_229e = pd.read_csv("/Users/dt14/Desktop/HCoV/REFs/229E/NCBI_virus/229E_metadata_09Sep26.csv")

metadata_229e_filt = filter_data(df= metadata_229e, 
                                ref_length = 27254,
                                min_coverage= 0.8
                                )

metadata_229e_filt.shape

metadata_229e_filt.shape


(549, 30)

- HCoV HKU1

In [7]:
# Importing the metadata df for HKU1
metadata_HKU1 = pd.read_csv("/Users/dt14/Desktop/HCoV/REFs/HKU1/NCBI_virus/HKU1_metadata_09Sep26.csv")

metadata_HKU1_filt = filter_data(df= metadata_HKU1, 
                                ref_length= 29926, 
                                min_coverage= 0.8
                                )

metadata_HKU1_filt.shape

(396, 30)

- HCoV NL63

In [8]:
# Importing the metadata df for NL63
metadata_NL63 = pd.read_csv("/Users/dt14/Desktop/HCoV/REFs/NL63/NCBI_virus/NL63_metadata_09Sep26.csv")

metadata_NL63_filt = filter_data(df= metadata_NL63, 
                                ref_length= 27553, 
                                min_coverage= 0.8
                                )

metadata_NL63_filt.shape

(417, 30)

- HCoV OC43

In [9]:
# Importing the metadata df for OC43
metadata_OC43 = pd.read_csv("/Users/dt14/Desktop/HCoV/REFs/OC43/NCBI_virus/OC43_metadata_09Sep26.csv")

metadata_OC43_filt = filter_data(df= metadata_OC43, 
                                ref_length= 30741, 
                                min_coverage= 0.8
                                )

metadata_OC43_filt.shape

(844, 30)

## 3 - Genotypes
Now I'll Load a new metadata with the genotypes for each HCoV species.

### 229E

In [10]:
from pandas import merge
genotyes_229E = pd.read_csv("/Users/dt14/Desktop/HCoV/REFs/229E/HCoV_229E_genotypes.csv")


In [11]:
metadata_229e_filt["Accession_clean"] = ( #creating a new column
    metadata_229e_filt["Accession"] #using the columns Accession
    .str.split(".") #splitting the string into tow valoes
    .str[0] #catching the first value
)

# merging the two dfs
metadata_229e_final = metadata_229e_filt.merge( 
    genotyes_229E,
    left_on="Accession_clean",
    right_on="accession_or_isolate",
    how="inner"
)

# concatenating the two DF again 
# some of the lines didn't match previously values so i'll cocatenate the two dfs
metadata_229e_final = pd.concat([metadata_229e_final, metadata_229e_filt])

# removing duplicated Accessions from the previous code.
metadata_229e_final = metadata_229e_final.drop_duplicates(subset=["Accession"])

### HKU1

In [12]:
genotypes_hku1 = pd.read_csv("/Users/dt14/Desktop/HCoV/REFs/HKU1/HCoV_HKU1_genotypes.csv")

metadata_HKU1_filt["Accession_clean"] = (
metadata_HKU1_filt["Accession"]
.str.split(".")
.str[0]
)

metadata_HKU1_final = metadata_HKU1_filt.merge(
    genotypes_hku1,
    left_on="Accession_clean",
    right_on="accession_or_isolate",
    how="inner"
)

metadata_HKU1_final = pd.concat([metadata_HKU1_final, metadata_HKU1_filt])

metadata_HKU1_final = metadata_HKU1_final.drop_duplicates(subset=["Accession"])

### NL63

In [13]:
genotypes_nl63 = pd.read_csv("/Users/dt14/Desktop/HCoV/REFs/NL63/HCoV_NL63_genotypes.csv")

metadata_NL63_filt["Accession_clean"] = (
metadata_NL63_filt["Accession"]
.str.split(".")
.str[0]
)

metadata_NL63_final = metadata_NL63_filt.merge(
    genotypes_nl63,
    left_on="Accession_clean",
    right_on="accession_or_isolate",
    how="inner"
)

metadata_NL63_final = pd.concat([metadata_NL63_final, metadata_NL63_filt])

metadata_NL63_final = metadata_NL63_final.drop_duplicates(subset=["Accession"])

### OC43

In [14]:
genotypes_oc43 = pd.read_csv("/Users/dt14/Desktop/HCoV/REFs/OC43/HCoV_OC43_genotypes.csv")

metadata_OC43_filt["Accession_clean"] = (
metadata_OC43_filt["Accession"]
.str.split(".")
.str[0]
)

metadata_OC43_final = metadata_OC43_filt.merge(
    genotypes_oc43,
    left_on="Accession_clean",
    right_on="accession_or_isolate",
    how="inner"
)

metadata_OC43_final = pd.concat([metadata_OC43_final, metadata_OC43_filt])

metadata_OC43_final = metadata_OC43_final.drop_duplicates(subset=["Accession"])
#metadata_OC43_final


## 4. Rename and filter the fasta file.

The header was created on the steps above. Now I'll need to filter and rename the fastas.

Here I'll export two files: 
1. List of Accessions to be used.
2. Table with two columns Accession and Header that will be used to rename fasta.

In [30]:
#saving the Accessions
metadata_229e_final["Accession"].to_csv("/Users/dt14/Desktop/HCoV/REFs/229E/229E_acc_keep.txt", sep="\t", index=False)
metadata_HKU1_final["Accession"].to_csv("/Users/dt14/Desktop/HCoV/REFs/HKU1/HKU1_acc_keep.txt", sep="\t", index=False)
metadata_NL63_final["Accession"].to_csv("/Users/dt14/Desktop/HCoV/REFs/NL63/NL63_acc_keep.txt", sep="\t", index=False)
metadata_OC43_final["Accession"].to_csv("/Users/dt14/Desktop/HCoV/REFs/OC43/OC43_acc_keep.txt", sep="\t", index=False)

#saving the dataframe to rename the fasta
metadata_229e_final[["Accession","name"]].to_csv("/Users/dt14/Desktop/HCoV/REFs/229E/229E_rename.txt", sep="\t", index=False, header=False)
metadata_HKU1_final[["Accession","name"]].to_csv("/Users/dt14/Desktop/HCoV/REFs/HKU1/HKU1_rename.txt", sep="\t", index=False, header=False)
metadata_NL63_final[["Accession","name"]].to_csv("/Users/dt14/Desktop/HCoV/REFs/NL63/NL63_rename.txt", sep="\t", index=False, header=False)
metadata_OC43_final[["Accession","name"]].to_csv("/Users/dt14/Desktop/HCoV/REFs/OC43/OC43_rename.txt", sep="\t", index=False, header=False)

#saving the matadata
cols = ["name","Accession","Date","Country","genotype","subgenotype"]
metadata_229e_final[cols].to_csv("/Users/dt14/Desktop/HCoV/REFs/229E/229E_metadata_final.txt", sep="\t", index=False, header=True)
metadata_HKU1_final[cols].to_csv("/Users/dt14/Desktop/HCoV/REFs/HKU1/HKU1_metadata_final.txt", sep="\t", index=False, header=False)
metadata_NL63_final[cols].to_csv("/Users/dt14/Desktop/HCoV/REFs/NL63/NL63_metadata_final.txt", sep="\t", index=False, header=False)
metadata_OC43_final[cols].to_csv("/Users/dt14/Desktop/HCoV/REFs/OC43/OC43_metadata_final.txt", sep="\t", index=False, header=False)


In [23]:
metadata_229e_final.columns

Index(['Accession', 'Assembly', 'SRA_Accession', 'BioSample', 'BioProject', 'Organism_Name', 'Species', 'Genus', 'Family', 'Genotype', 'Isolate', 'Segment', 'GenBank_Title', 'Length',
       'Nuc_Completeness', 'Geo_Location', 'Country', 'USA', 'Host', 'Tissue_Specimen_Source', 'Submitters', 'Organization', 'Org_location', 'Publications', 'Collection_Date', 'Release_Date',
       'Molecule_type', 'Date', 'name', 'coverage', 'Accession_clean', 'accession_or_isolate', 'genotype', 'subgenotype'],
      dtype='str')

In [18]:
# filtering the final genomes
!seqtk subseq \
    /Users/dt14/Desktop/HCoV/REFs/229E/NCBI_virus/229E_sequences_09Sep26.fasta \
    /Users/dt14/Desktop/HCoV/REFs/229E/229E_acc_keep.txt \
    > /Users/dt14/Desktop/HCoV/REFs/229E/NCBI_virus/229E_sequences_09Sep26_filtered.fasta \    

!seqtk subseq \
    /Users/dt14/Desktop/HCoV/REFs/HKU1/NCBI_virus/HKU1_sequences_09Sep26.fasta \
    /Users/dt14/Desktop/HCoV/REFs/HKU1/HKU1_acc_keep.txt \
    > /Users/dt14/Desktop/HCoV/REFs/HKU1/NCBI_virus/HKU1_sequences_09Sep26_filtered.fasta \

!seqtk subseq \
    /Users/dt14/Desktop/HCoV/REFs/NL63/NCBI_virus/NL63_sequences_09Sep26.fasta \
    /Users/dt14/Desktop/HCoV/REFs/NL63/NL63_acc_keep.txt \
    > /Users/dt14/Desktop/HCoV/REFs/NL63/NCBI_virus/NL63_sequences_09Sep26_filtered.fasta \

!seqtk subseq \
    /Users/dt14/Desktop/HCoV/REFs/OC43/NCBI_virus/OC43_sequences_09Sep26.fasta \
    /Users/dt14/Desktop/HCoV/REFs/OC43/OC43_acc_keep.txt \
    > /Users/dt14/Desktop/HCoV/REFs/OC43/NCBI_virus/OC43_sequences_09Sep26_filtered.fasta \
        

## 5. Rename and filter the fasta file.

In [22]:
# renaming the fasta files

!~/miniconda3/bin/conda run -n tools seqkit replace \
    -p '^(\S+).*' \
    -r '{kv}' \
    -k /Users/dt14/Desktop/HCoV/REFs/229E/229E_rename.txt \
    -K \
    /Users/dt14/Desktop/HCoV/REFs/229E/NCBI_virus/229E_sequences_09Sep26_filtered.fasta \
    -o /Users/dt14/Desktop/HCoV/REFs/229E/NCBI_virus/229E_sequences_09Sep26_formated.fasta


!~/miniconda3/bin/conda run -n tools seqkit replace \
    -p '^(\S+).*' \
    -r '{kv}' \
    -k /Users/dt14/Desktop/HCoV/REFs/HKU1/HKU1_rename.txt \
    -K \
    /Users/dt14/Desktop/HCoV/REFs/HKU1/NCBI_virus/HKU1_sequences_09Sep26_filtered.fasta \
    -o /Users/dt14/Desktop/HCoV/REFs/HKU1/NCBI_virus/HKU1_sequences_09Sep26_formated.fasta


!~/miniconda3/bin/conda run -n tools seqkit replace \
    -p '^(\S+).*' \
    -r '{kv}' \
    -k /Users/dt14/Desktop/HCoV/REFs/NL63/NL63_rename.txt \
    -K \
    /Users/dt14/Desktop/HCoV/REFs/NL63/NCBI_virus/NL63_sequences_09Sep26_filtered.fasta \
    -o /Users/dt14/Desktop/HCoV/REFs/NL63/NCBI_virus/NL63_sequences_09Sep26_formated.fasta


!~/miniconda3/bin/conda run -n tools seqkit replace \
    -p '^(\S+).*' \
    -r '{kv}' \
    -k /Users/dt14/Desktop/HCoV/REFs/OC43/OC43_rename.txt \
    -K \
    /Users/dt14/Desktop/HCoV/REFs/OC43/NCBI_virus/OC43_sequences_09Sep26_filtered.fasta \
    -o /Users/dt14/Desktop/HCoV/REFs/OC43/NCBI_virus/OC43_sequences_09Sep26_formated.fasta

[INFO] read key-value file: /Users/dt14/Desktop/HCoV/REFs/229E/229E_rename.txt
[INFO] 549 pairs of key-value loaded

[INFO] read key-value file: /Users/dt14/Desktop/HCoV/REFs/HKU1/HKU1_rename.txt
[INFO] 396 pairs of key-value loaded

[INFO] read key-value file: /Users/dt14/Desktop/HCoV/REFs/NL63/NL63_rename.txt
[INFO] 417 pairs of key-value loaded

[INFO] read key-value file: /Users/dt14/Desktop/HCoV/REFs/OC43/OC43_rename.txt
[INFO] 844 pairs of key-value loaded

